# Summarization Training with Checkpoint Resume
This notebook trains T5 and automatically resumes from the latest checkpoint in ./results if one exists.

In [ ]:
import os
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

In [ ]:
dataset = load_dataset('json', data_files='summarization_dataset.json')
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

def preprocess(example):
    input_text = str(example['input'])
    target_text = example['target']

    inputs = tokenizer(input_text, padding='max_length', truncation=True, max_length=128)
    labels = tokenizer(target_text, padding='max_length', truncation=True, max_length=64)
    inputs['labels'] = labels['input_ids']
    return inputs

dataset = dataset.map(preprocess)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_strategy='steps',
    save_steps=25,
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
)

last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)

print(f'Last checkpoint: {last_checkpoint}')
trainer.train(resume_from_checkpoint=last_checkpoint)
model.save_pretrained('./summarization_model')
tokenizer.save_pretrained('./summarization_model')